#Spark SQL

In [0]:
df_csv=spark.read.format("csv")\
    .option("header",True)\
    .option("inferSchema",True)\
    .load("/Volumes/sparkcatalog/raw/source/raw_orders/orders.csv")


display(df_csv)

###Temporary View

In [0]:
df_csv.createOrReplaceTempView("csv_view")

In [0]:
display(spark.sql("select * from csv_view"))

In [0]:
# display(spark.sql(''' 
#           select * from csv_view
#           where order_id in (1001,1002)
#           '''))

display(
spark.sql('''
          select * from csv_view
          where order_id in (1001,1002,1003)
          ''')
)

###Create Dataframe on top sql

In [0]:
df_sql=spark.sql('''
                 select * from csv_view
                 where order_id in (1001,1002)
                 ''')

In [0]:
display(df_sql)

###Global Temp View

In [0]:
df_csv.createOrReplaceGlobalTempView("csv_view1")

###DDL Commands

In [0]:
%fs head "/Volumes/sparkcatalog/raw/source/raw_orders/orders.csv"

In [0]:
spark.sql('''
          create table if not exists sparkcatalog.raw.emp_tab
          (
              id int,
              name string,
              age int
          )
          ''')

In [0]:
spark.sql('''
          insert into sparkcatalog.raw.emp_tab
          (id, name, age)
          values
          (100, 'John', 30),
          (101, 'Jane', 25),
          (102, 'Bob', 35)
          ''')

In [0]:
display(
    spark.sql('''
              select * from sparkcatalog.raw.emp_tab
              ''')
)

In [0]:
spark.sql('''
          create table if not exists sparkcatalog.raw.emp_address
          (
              id int,
              address string
          )
          ''')

spark.sql('''
          insert into sparkcatalog.raw.emp_address
          (id, address)
          values
          (100, '123 Main St'),
          (101, '456 Oak Ave'),
          (102, '789 Pine Rd')
          ''')


In [0]:
display(
    spark.sql('''
              select * from sparkcatalog.raw.emp_address
              ''')
)

###Join using Spark SQL

In [0]:
display(
    spark.sql('''
              select *
              from sparkcatalog.raw.emp_tab e
              left join sparkcatalog.raw.emp_address a
              on e.id = a.id
              ''')
)

###Upsert command using Spark SQL

In [0]:
data=[(100,'John',30),(101,'Jane',36),(103,'Bobi',35)]

df_new=spark.createDataFrame(data, "id int,name string, age int")

display(df_new)



In [0]:
df_new.createOrReplaceTempView("new_emp_data")


#applying upsert wit merge command

spark.sql('''
          merge into sparkcatalog.raw.emp_tab tgt
          using new_emp_data src
          on tgt.id = src.id
          when matched then update set *
          when not matched then insert *
          ''')

display(
    spark.sql('''
              select * from sparkcatalog.raw.emp_tab
              ''')
)


###MergeInto API (Spark 4.0+)

In [0]:
data=[(100, 'John', 30), (101, 'Jane', 40), (105, 'Robin', 38)]

df_new2=spark.createDataFrame(data,("id int, name string, age int"))

display(df_new2)

In [0]:
from pyspark.sql.functions import col

(df_new2.alias("src").mergeInto("sparkcatalog.raw.emp_tab", col("src.id")==col("sparkcatalog.raw.emp_tab.id"))
    .whenMatched().updateAll()
    .whenNotMatched().insertAll()
    .merge()
)

In [0]:
display(
    spark.sql('''
              select * from sparkcatalog.raw.emp_tab
              ''')
)

In [0]:
%sql
create table sparkcatalog.raw.part_table
(
    id int,
    name string
)
using delta
partitioned by (id)

In [0]:
%sql

insert into sparkcatalog.raw.part_table
( id, name)
values
(1, 'a'),
(2, 'b'),
(3, 'c'),
(4, 'd'),
(5, 'e')


In [0]:
%sql
explain 
select * from sparkcatalog.raw.part_table

### SQL Funcations

In [0]:
%sql
select * from sparkcatalog.raw.part_table;

In [0]:
%sql
with inner_query as(
select *,
    case when id%2==0 then 'even' else 'odd' end as even_odd
from sparkcatalog.raw.part_table
)
select * from inner_query;

###SQL Scripting statements

###For Statments

In [0]:
%sql
begin
    declare sum int default 0;
    sumNumbers: for row as select num from range(1,20) as t(num) do
        if num > 10 then
            leave sumNumbers;
        elseif num % 2= 0 then 
            iterate sumNumbers;
        end if;
        set sum=sum+row.num;
        end for sumNumbers;
        values(sum);
end;
        

###If Statment

In [0]:
%sql
BEGIN
    DECLARE choice DOUBLE DEFAULT 3.9;
    DECLARE result STRING;

    IF choice < 2 THEN
        SET result = 'one fish';
    ELSEIF choice < 3 THEN
        SET result = 'two fish';
    ELSEIF choice < 4 THEN
        SET result = 'red fish';
    ELSEIF choice < 5 OR choice IS NULL THEN
        SET result = 'blue fish';
    ELSE
        SET result = 'no fish';
    END IF;

    VALUES(result);
END;


###Case

In [0]:
%sql
begin 
    declare choice int default 3;
    declare result string;
    case choice
        when 1 then 
            values('one fish');
        when 2 then 
            values('two fish');
        when 3 then 
            values('red fish');
        when 4 then
            values('blue fish');
        else    
            values ('no fish');
    end case;
end;

###Spark sql Auxiliary Statements

#Describe

In [0]:
%sql
describe database sparkcatalog.raw

In [0]:
%sql
describe table sparkcatalog.raw.part_table

In [0]:
%sql
describe table extended  sparkcatalog.raw.part_table

In [0]:
%sql
show create table sparkcatalog.raw.part_table;

In [0]:
%sql
describe query select * from sparkcatalog.raw.part_table;

###Refresh

In [0]:
%sql
refresh table sparkcatalog.raw.part_table;

###Show

In [0]:
%sql
show databases like 'sparkcatalog';

In [0]:
%sql
show catalogs;

In [0]:
display(
    spark.sql('''
              show catalogs
              ''')
)

In [0]:
%sql
show schemas in sparkcatalog;

In [0]:
display(
    spark.sql('''
              show schemas in sparkcatalog
              ''')
)

In [0]:
%sql
show tables in sparkcatalog.raw;

In [0]:
display(
    spark.sql(
        '''
        show tables in sparkcatalog.raw
        '''
    )
)

In [0]:
# 1. List catalogs
catalogs = spark.sql("SHOW CATALOGS").collect()
for c in catalogs:
    print(c[0])

# 2. Pick a catalog, list schemas
catalog_name = "sparkcatalog"
schemas = spark.sql(f"SHOW SCHEMAS IN {catalog_name}").collect()
for s in schemas:
    print(s[0])

# 3. Pick a schema, list tables
schema_name = "raw"   # just the schema, not catalog.schema
tables = spark.sql(f"SHOW TABLES IN {catalog_name}.{schema_name}").collect()
for t in tables:
    print(t.tableName)

# 4. Describe / preview each table
for t in tables:
    full_name = f"{catalog_name}.{schema_name}.{t.tableName}"
    try:
        print(f"--- {full_name} ---")
        display(spark.sql(f"DESCRIBE TABLE {full_name}"))
        display(spark.table(full_name).limit(5))
    except Exception as e:
        print(f"Skipping {full_name}: {e}")

In [0]:
%sql
SHOW TABLES IN sparkcatalog.raw LIKE 'part_table';

In [0]:
%sql
use sparkcatalog.raw;

show table extended like 'part_table';

In [0]:
%sql
show tblproperties sparkcatalog.raw.part_table

In [0]:
%sql
show partitions sparkcatalog.raw.part_table;